In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df_raw = pd.read_csv("Crime_Data_from_2020_to_Present_20260228.csv")
df_raw.head()

In [ ]:
# Step 1: 查看所有列名
df_raw.columns

In [ ]:
# 复制一份数据
df = df_raw.copy()

# 统一列名格式（全部小写 + 下划线 + 去空格）
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.columns

In [ ]:
df[["date_occ", "crm_cd_desc", "lat", "lon"]].head()

In [ ]:
# Step 2: 类型转换

# 1️⃣ 转换 date_occ 为 datetime
df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")

# 2️⃣ 转换 lat / lon 为 float
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# 3️⃣ 清理 crime 描述（统一大写，去空格）
df["crm_cd_desc"] = df["crm_cd_desc"].astype(str).str.strip().str.upper()

# 查看转换后的类型
df[["date_occ", "lat", "lon"]].dtypes

In [ ]:
df[["date_occ", "lat", "lon"]].isna().sum()

In [ ]:
# Step 3: 删除无效坐标 + LA bounding box 粗过滤

rows_before = len(df)

# 1) 删除 0 坐标（有些数据会有 0，但你的 NA=0，不代表没有 0）
df = df[(df["lat"] != 0) & (df["lon"] != 0)]

rows_after_zero = len(df)

# 2) LA 粗范围过滤（宁可宽一点，避免误删）
min_lat = 33.2
max_lat = 34.95
min_lon = -119.10
max_lon = -117.40

df = df[(df["lat"] >= min_lat) & (df["lat"] <= max_lat)]
df = df[(df["lon"] >= min_lon) & (df["lon"] <= max_lon)]

rows_after_bbox = len(df)

rows_before, rows_after_zero, rows_after_bbox

In [ ]:
df[["lat","lon"]].describe()

In [ ]:
# Step 4: 筛选 2022–2024 + 添加 year

df["year"] = df["date_occ"].dt.year

df = df[(df["year"] >= 2022) & (df["year"] <= 2024)]

df["year"].value_counts().sort_index()

In [ ]:
# Step 5: 按 dr_no 去重

rows_before = len(df)

df["dr_no"] = df["dr_no"].astype(str).str.strip()
df = df.drop_duplicates(subset=["dr_no"])

rows_after = len(df)

rows_before, rows_after, rows_before - rows_after

In [ ]:
# Step 6: 分类 + 只保留 relevant

violent_keywords = ["HOMICIDE", "ASSAULT", "ROBBERY"]
property_keywords = ["BURGLARY", "THEFT", "CRIMINAL DAMAGE"]

def classify_crime(desc):
    desc = str(desc).upper()
    for k in violent_keywords:
        if k in desc:
            return "violent"
    for k in property_keywords:
        if k in desc:
            return "property"
    return "other"

df["crime_type"] = df["crm_cd_desc"].apply(classify_crime)

df["crime_type"].value_counts()

In [ ]:
df_rel = df[df["crime_type"] != "other"].copy()

df_rel["crime_type"].value_counts()

In [ ]:
df_rel_out = df_rel[[
    "date_occ",
    "year",
    "crime_type",
    "lat",
    "lon"
]].copy()

df_rel_out.to_parquet("crime_points_2022_2024_relevant.parquet", index=False)
df_rel_out.to_csv("crime_points_2022_2024_relevant.csv", index=False)

len(df_rel_out)

In [ ]:
df_rel_out.head()

In [ ]:
df_rel_out.info()

In [ ]:
df_rel_out.describe()

In [ ]:
# Spatial Join

In [ ]:
!pip install geopandas

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import geopandas as gpd

tracts = gpd.read_file("tl_2024_06_tract.shp")
tracts.head()

In [ ]:
tracts.columns

In [ ]:
la_tracts = tracts[tracts["COUNTYFP"] == "037"].copy()
la_tracts = la_tracts[["GEOID", "geometry"]]

len(la_tracts)

In [ ]:
la_tracts.crs

In [ ]:
la_tracts = la_tracts.to_crs("EPSG:4326")
la_tracts.crs

In [ ]:
crime_gdf = gpd.GeoDataFrame(
    df_rel_out,
    geometry=gpd.points_from_xy(df_rel_out["lon"], df_rel_out["lat"]),
    crs="EPSG:4326"
)

crime_gdf.head()

In [ ]:
crime_with_tract = gpd.sjoin(
    crime_gdf,
    la_tracts,
    how="inner",
    predicate="within"
)

crime_with_tract[["date_occ", "year", "crime_type", "lat", "lon", "GEOID"]].head()

In [ ]:
len(df_rel_out), len(crime_with_tract)

In [ ]:
tract_crime = (
    crime_with_tract
    .groupby(["GEOID", "crime_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

tract_crime["total_relevant"] = tract_crime["property"] + tract_crime["violent"]

tract_crime.head()

In [ ]:
tract_crime.to_csv("la_tract_crime_2022_2024_relevant_counts.csv", index=False)
tract_crime.to_parquet("la_tract_crime_2022_2024_relevant_counts.parquet", index=False)

tract_crime.shape

In [ ]:
# 把 0 crime tract 补回来

In [ ]:
tract_crime_full = (
    la_tracts[["GEOID"]]
    .merge(tract_crime, on="GEOID", how="left")
    .fillna(0)
)

tract_crime_full.head()

In [ ]:
tract_crime_full.shape

In [ ]:
tract_crime_full[["property", "violent", "total_relevant"]] = \
    tract_crime_full[["property", "violent", "total_relevant"]].astype(int)

tract_crime_full.dtypes

In [ ]:
tract_crime_full.head()

In [ ]:
# ACS total population

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

acs22 = pd.read_csv("ACSDT5Y2022.B01003-Data.csv")
acs23 = pd.read_csv("ACSDT5Y2023.B01003-Data.csv")
acs24 = pd.read_csv("ACSDT5Y2024.B01003-Data.csv")

acs22.head()

In [ ]:
acs22.columns

In [ ]:
acs22 = acs22.iloc[1:].copy()
acs23 = acs23.iloc[1:].copy()
acs24 = acs24.iloc[1:].copy()

acs22.head()

In [ ]:
acs22["GEOID"] = acs22["GEO_ID"].astype(str).str[-11:]
acs23["GEOID"] = acs23["GEO_ID"].astype(str).str[-11:]
acs24["GEOID"] = acs24["GEO_ID"].astype(str).str[-11:]

In [ ]:
acs22["pop_2022"] = pd.to_numeric(acs22["B01003_001E"], errors="coerce")
acs23["pop_2023"] = pd.to_numeric(acs23["B01003_001E"], errors="coerce")
acs24["pop_2024"] = pd.to_numeric(acs24["B01003_001E"], errors="coerce")

In [ ]:
pop22 = acs22[["GEOID", "pop_2022"]].copy()
pop23 = acs23[["GEOID", "pop_2023"]].copy()
pop24 = acs24[["GEOID", "pop_2024"]].copy()

In [ ]:
acs_pop = pop22.merge(pop23, on="GEOID", how="outer")
acs_pop = acs_pop.merge(pop24, on="GEOID", how="outer")

acs_pop.head()

In [ ]:
acs_pop_la = acs_pop[acs_pop["GEOID"].str.startswith("06037")].copy()

acs_pop_la.shape

In [ ]:
acs_pop_la["pop_avg_22_24"] = (
    acs_pop_la["pop_2022"] +
    acs_pop_la["pop_2023"] +
    acs_pop_la["pop_2024"]
) / 3

acs_pop_la.head()

In [ ]:
crime_acs = tract_crime_full.merge(
    acs_pop_la[["GEOID", "pop_avg_22_24"]],
    on="GEOID",
    how="left"
)

crime_acs.head()

In [ ]:
crime_acs["violent_per_1000"] = crime_acs["violent"] / crime_acs["pop_avg_22_24"] * 1000
crime_acs["property_per_1000"] = crime_acs["property"] / crime_acs["pop_avg_22_24"] * 1000

crime_acs[["GEOID", "violent_per_1000", "property_per_1000"]].head()

In [ ]:
crime_acs["pop_avg_22_24"].isna().sum()

In [ ]:
crime_acs["pop_avg_22_24"].describe()

In [ ]:
crime_acs_valid = crime_acs[crime_acs["pop_avg_22_24"] >= 500].copy()

crime_acs_valid.shape

In [ ]:
violent_cut = crime_acs_valid["violent_per_1000"].quantile(0.60)
property_cut = crime_acs_valid["property_per_1000"].quantile(0.60)

violent_cut, property_cut

In [ ]:
crime_acs_valid["crime_ok_60"] = (
    (crime_acs_valid["violent_per_1000"] <= violent_cut) &
    (crime_acs_valid["property_per_1000"] <= property_cut)
)

crime_acs_valid["crime_ok_60"].value_counts()

In [ ]:
import pandas as pd

def acs_clean_one_year(df_raw, year, value_col, new_col_name):
    """
    Extract GEOID and one ACS estimate column from a raw ACS file.
    """
    df = df_raw.iloc[1:].copy()

    # 1) GEOID: Extract the last 11 digits from GEO_ID (tract ID)
    df["GEOID"] = df["GEO_ID"].astype(str).str[-11:]

    # 2) Convert value column to numeric
    df[new_col_name] = pd.to_numeric(df[value_col], errors="coerce")

    # 3) Keep only required columns
    df = df[["GEOID", new_col_name]].copy()

    return df

In [ ]:
def acs_merge_3years(df22_raw, df23_raw, df24_raw, value_col, base_name):
    """
    Merge 2022/2023/2024 of the same ACS table into a wide format:
    GEOID | {base_name}_2022 | {base_name}_2023 | {base_name}_2024 | {base_name}_avg_22_24
    """
    d22 = acs_clean_one_year(df22_raw, 2022, value_col, f"{base_name}_2022")
    d23 = acs_clean_one_year(df23_raw, 2023, value_col, f"{base_name}_2023")
    d24 = acs_clean_one_year(df24_raw, 2024, value_col, f"{base_name}_2024")

    out = d22.merge(d23, on="GEOID", how="outer")
    out = out.merge(d24, on="GEOID", how="outer")

    out[f"{base_name}_avg_22_24"] = (
        out[f"{base_name}_2022"] +
        out[f"{base_name}_2023"] +
        out[f"{base_name}_2024"]
    ) / 3

    # Keep only LA County tracts (tracts starting with 06037)
    out = out[out["GEOID"].astype(str).str.startswith("06037")].copy()

    return out

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
inc22 = pd.read_csv("ACSDT5Y2022.B19013-Data.csv")
inc23 = pd.read_csv("ACSDT5Y2023.B19013-Data.csv")
inc24 = pd.read_csv("ACSDT5Y2024.B19013-Data.csv")

In [ ]:
acs_income_la = acs_merge_3years(
    inc22,
    inc23,
    inc24,
    value_col="B19013_001E",
    base_name="income"
)

acs_income_la.head()

In [ ]:
crime_features = crime_acs_valid.merge(
    acs_income_la[["GEOID", "income_avg_22_24"]],
    on="GEOID",
    how="left"
)

crime_features.head()

In [ ]:
# education

In [ ]:
edu22 = pd.read_csv("ACSDT5Y2022.B15003-Data.csv")
edu23 = pd.read_csv("ACSDT5Y2023.B15003-Data.csv")
edu24 = pd.read_csv("ACSDT5Y2024.B15003-Data.csv")

In [ ]:
def acs_clean_ba_plus_rate(df_raw, year, new_col_name):
    df = df_raw.iloc[1:].copy()

    df["GEOID"] = df["GEO_ID"].astype(str).str[-11:]

    total_25 = pd.to_numeric(df["B15003_001E"], errors="coerce")

    ba = pd.to_numeric(df["B15003_022E"], errors="coerce")
    ma = pd.to_numeric(df["B15003_023E"], errors="coerce")
    prof = pd.to_numeric(df["B15003_024E"], errors="coerce")
    phd = pd.to_numeric(df["B15003_025E"], errors="coerce")

    ba_plus = ba + ma + prof + phd

    df[new_col_name] = ba_plus / total_25

    df = df[["GEOID", new_col_name]].copy()

    return df

In [ ]:
edu_22 = acs_clean_ba_plus_rate(edu22, 2022, "ba_plus_rate_2022")
edu_23 = acs_clean_ba_plus_rate(edu23, 2023, "ba_plus_rate_2023")
edu_24 = acs_clean_ba_plus_rate(edu24, 2024, "ba_plus_rate_2024")

acs_edu_la = edu_22.merge(edu_23, on="GEOID", how="outer")
acs_edu_la = acs_edu_la.merge(edu_24, on="GEOID", how="outer")

acs_edu_la["ba_plus_rate_avg_22_24"] = (
    acs_edu_la[
        ["ba_plus_rate_2022",
         "ba_plus_rate_2023",
         "ba_plus_rate_2024"]
    ].mean(axis=1)
)

acs_edu_la = acs_edu_la[acs_edu_la["GEOID"].astype(str).str.startswith("06037")].copy()

acs_edu_la.head()

In [ ]:
crime_features = crime_features.merge(
    acs_edu_la[["GEOID", "ba_plus_rate_avg_22_24"]],
    on="GEOID",
    how="left"
)

In [ ]:
crime_features = crime_features.loc[:, ~crime_features.T.duplicated()]

In [ ]:
crime_features = crime_features.rename(
    columns={"ba_plus_rate_avg_22_24_x": "ba_plus_rate_avg_22_24"}
)

In [ ]:
crime_features.head()

加入 land area → 计算人口密度
从 TIGER 读出 LA tracts 的 ALAND（做成小表）

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
os.listdir()

In [ ]:
import geopandas as gpd

tracts_gdf = gpd.read_file("tl_2024_06_tract.shp")

In [ ]:
tracts_gdf.head()

In [ ]:
la_tracts = tracts_gdf[tracts_gdf["COUNTYFP"] == "037"].copy()

In [ ]:
la_tracts.shape

In [ ]:
aland_table = la_tracts[["GEOID", "ALAND"]].copy()

In [ ]:
aland_table["GEOID"] = aland_table["GEOID"].astype(str)
crime_features["GEOID"] = crime_features["GEOID"].astype(str)

In [ ]:
crime_features = crime_features.merge(
    aland_table,
    on="GEOID",
    how="left",
    validate="one_to_one"
)

In [ ]:
crime_features["land_sqkm"] = crime_features["ALAND"] / 1_000_000

In [ ]:
crime_features["pop_density"] = (
    crime_features["pop_avg_22_24"] / crime_features["land_sqkm"]
)

In [ ]:
crime_features[
    ["GEOID","ALAND","land_sqkm","pop_avg_22_24","pop_density"]
].head()

In [ ]:
tract_acs = crime_features[
    [
        "GEOID",
        "income_avg_22_24",
        "ba_plus_rate_avg_22_24",
        "pop_avg_22_24",
        "pop_density"
    ]
].copy()

In [ ]:
tract_acs = tract_acs.rename(columns={
    "income_avg_22_24": "income",
    "ba_plus_rate_avg_22_24": "ba_rate",
    "pop_avg_22_24": "pop"
})

In [ ]:
tract_acs.head()

In [ ]:
tract_acs.to_csv("tract_acs.csv", index=False)

In [ ]:
import os
os.listdir()

In [ ]:
import pandas as pd

pd.read_csv("tract_acs.csv").head()

In [ ]:
from google.colab import files
files.download("tract_acs.csv")

In [ ]:
tract_crime = crime_features[
    [
        "GEOID",
        "violent_per_1000",
        "property_per_1000",
        "crime_ok_60"
    ]
].copy()

tract_crime.to_csv("tract_crime.csv", index=False)

In [ ]:
os.listdir()

In [ ]:
import pandas as pd

pd.read_csv("tract_crime.csv").head()

In [ ]:
from google.colab import files
files.download("tract_crime.csv")

osm数据

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
[x for x in os.listdir() if x.startswith("points.")]

In [ ]:
import geopandas as gpd
points = gpd.read_file("points.shp")
points.head()

In [ ]:
points.columns

In [ ]:
retail_types = [
    "restaurant",
    "fast_food",
    "cafe",
    "bar",
    "pharmacy",
    "bank",
    "atm",
    "fuel",
    "ice_cream",
    "dentist",
    "clinic",
    "post_office",
    "post_box",
    "bicycle_rental",
    "vending_machine"
]

In [ ]:
retail_points = points[points["type"].isin(retail_types)].copy()

In [ ]:
retail_points["type"].value_counts()

In [ ]:
retail_points.shape

In [ ]:
retail_points = retail_points.to_crs(la_tracts.crs)

In [ ]:
la_tracts = tracts_gdf[tracts_gdf["COUNTYFP"] == "037"].copy()

In [ ]:
points_tract = gpd.sjoin(
    retail_points,
    la_tracts[["GEOID", "geometry"]],
    how="inner",
    predicate="within"
)

In [ ]:
points_tract.head()

In [ ]:
retail_counts = (
    points_tract.groupby("GEOID")
    .size()
    .reset_index(name="retail_count")
)

In [ ]:
retail_counts = retail_counts.merge(
    crime_features[["GEOID", "land_sqkm"]],
    on="GEOID",
    how="left"
)

In [ ]:
retail_counts.columns

In [ ]:
retail_counts["retail_density"] = (
    retail_counts["retail_count"] / retail_counts["land_sqkm"]
)

In [ ]:
retail_counts.head()

In [ ]:
tract_osm = retail_counts[["GEOID", "retail_density"]].copy()
tract_osm.to_csv("tract_osm.csv", index=False)

In [ ]:
tract_osm.head()
tract_osm.shape

In [ ]:
from google.colab import files
files.download("tract_osm.csv")

Whole Foods Excel

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
wf_df = pd.read_excel("LA Whole Foods Location with Latitude&Longitude.xlsx")
wf_df.head()

In [ ]:
wf_df = wf_df.rename(columns={
    "Location": "location",
    "Latitude": "lat",
    "Longitude": "lon"
})

wf_df.head()

In [ ]:
import geopandas as gpd

wf_gdf = gpd.GeoDataFrame(
    wf_df.copy(),
    geometry=gpd.points_from_xy(wf_df["lon"], wf_df["lat"]),
    crs="EPSG:4326"
)

wf_gdf.head()

In [ ]:
la_tracts = tracts_gdf[tracts_gdf["COUNTYFP"] == "037"].copy()

In [ ]:
wf_join = gpd.sjoin(
    wf_gdf,
    la_tracts[["GEOID", "geometry"]],
    how="inner",
    predicate="within"
)

wf_join[["location", "lat", "lon", "GEOID"]].head()

In [ ]:
wf_tracts = wf_join[["GEOID"]].drop_duplicates().copy()
wf_tracts["has_wf"] = 1

wf_tracts.head()

In [ ]:
all_tracts = crime_features[["GEOID"]].drop_duplicates().copy()

tract_wf = all_tracts.merge(
    wf_tracts,
    on="GEOID",
    how="left"
)

tract_wf["has_wf"] = tract_wf["has_wf"].fillna(0).astype(int)

tract_wf.head()

In [ ]:
tract_wf["has_wf"].value_counts()

In [ ]:
tract_wf["has_wf"].sum()

In [ ]:
tract_wf.to_csv("tract_wf.csv", index=False)

In [ ]:
tract_wf["has_wf"].value_counts()
tract_wf.shape

In [ ]:
from google.colab import files
files.download("tract_wf.csv")